# Lab 24: Linear Algebra and Differential Equations

This independent-study notebook accompanies Chapter 24. It includes worked solutions and similar practice questions.

Topics:

- matrix exponentials;
- eigenvalue solutions;
- stability;
- Jordan blocks;
- graph heat equations;
- forward Euler time stepping.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm, eig
np.set_printoptions(precision=4, suppress=True)


## 1. Solving $\mathbf{x}'=A\mathbf{x}$ using $e^{At}$

The exact solution is

$$
\mathbf{x}(t)=e^{At}\mathbf{x}_0.
$$


In [ ]:
A = np.array([[2, 1],
              [1, 2]], dtype=float)
x0 = np.array([4, 2], dtype=float)

for t in [0, 0.25, 0.5, 1.0]:
    print(t, expm(A*t) @ x0)


### Similar practice 1

Use Python to solve

$$
\mathbf{x}'=\begin{bmatrix}1&0\\0&-2\end{bmatrix}\mathbf{x},
\qquad
\mathbf{x}(0)=\begin{bmatrix}3\\4\end{bmatrix}.
$$

Try it before opening the solution below.

<details>
<summary>Solution</summary>

Since the matrix is diagonal,

$$
\mathbf{x}(t)=\begin{bmatrix}3e^t\\4e^{-2t}\end{bmatrix}.
$$

The Python check is below.

</details>


In [ ]:
A1 = np.array([[1, 0],
               [0, -2]], dtype=float)
x01 = np.array([3, 4], dtype=float)
for t in [0, 1, 2]:
    print(t, expm(A1*t) @ x01)


## 2. Eigenbasis solution

If $A=PDP^{-1}$, then

$$
e^{At}=Pe^{Dt}P^{-1}.
$$


In [ ]:
vals, vecs = eig(A)
print("eigenvalues:", vals)
print("eigenvectors:")
print(vecs)

D = np.diag(vals)
P = vecs
print("Check A = P D P^{-1}:")
print(P @ D @ np.linalg.inv(P))


## 3. Stability from eigenvalues

The zero equilibrium is asymptotically stable if every eigenvalue has negative real part.


In [ ]:
systems = {
    "stable node": np.array([[-2, 0], [0, -5]], dtype=float),
    "saddle": np.array([[1, 0], [0, -3]], dtype=float),
    "stable spiral": np.array([[-1, -4], [1, -1]], dtype=float),
    "center": np.array([[0, -2], [2, 0]], dtype=float),
}

for name, M in systems.items():
    eigvals = np.linalg.eigvals(M)
    print(name, eigvals, "max real part =", np.max(np.real(eigvals)))


### Similar practice 2

Classify the equilibrium for

$$
A=\begin{bmatrix}-1&-4\\1&-1\end{bmatrix}.
$$

<details>
<summary>Solution</summary>

The eigenvalues are $-1\pm 2i$. Both have negative real part, so the equilibrium is asymptotically stable. The imaginary part means the solutions spiral.

</details>


## 4. Plotting trajectories


In [ ]:
def trajectory(A, x0, T=5, n=300):
    ts = np.linspace(0, T, n)
    xs = np.array([expm(A*t) @ x0 for t in ts])
    return ts, xs

A_spiral = np.array([[-1, -4], [1, -1]], dtype=float)
ts, xs = trajectory(A_spiral, np.array([2.0, 0.0]), T=8)
plt.figure()
plt.plot(xs[:,0], xs[:,1])
plt.scatter([xs[0,0]], [xs[0,1]], label="start")
plt.scatter([0], [0], label="equilibrium")
plt.axis("equal")
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Stable spiral trajectory")
plt.legend()
plt.show()


## 5. Jordan block and polynomial factors

For

$$
J=\begin{bmatrix}\lambda&1\\0&\lambda\end{bmatrix},
$$

we have

$$
e^{Jt}=e^{\lambda t}\begin{bmatrix}1&t\\0&1\end{bmatrix}.
$$


In [ ]:
lam = 3
J = np.array([[lam, 1], [0, lam]], dtype=float)
for t in [0, 0.5, 1.0]:
    formula = np.exp(lam*t) * np.array([[1, t], [0, 1]], dtype=float)
    print("t=", t)
    print("expm(Jt)=
", expm(J*t))
    print("formula=
", formula)


### Similar practice 3

Compute $e^{Jt}$ for

$$
J=\begin{bmatrix}2&1\\0&2\end{bmatrix}.
$$

<details>
<summary>Solution</summary>

$$
e^{Jt}=e^{2t}\begin{bmatrix}1&t\\0&1\end{bmatrix}.
$$

</details>


## 6. Graph heat equation

For the path graph $1-2-3$,

$$
L=\begin{bmatrix}1&-1&0\\-1&2&-1\\0&-1&1\end{bmatrix}.
$$

The graph heat equation is

$$
\mathbf u'=-L\mathbf u.
$$


In [ ]:
L = np.array([[ 1, -1,  0],
              [-1,  2, -1],
              [ 0, -1,  1]], dtype=float)
u0 = np.array([10, 0, 0], dtype=float)
print("eigenvalues of L:", np.linalg.eigvalsh(L))
for t in [0, 0.1, 1, 10]:
    print(t, expm(-L*t) @ u0)
print("average =", np.mean(u0))


### Similar practice 4

Try initial condition $u(0)=(0,6,12)^T$. What is the long-time limit?

<details>
<summary>Solution</summary>

The average is $(0+6+12)/3=6$. Since the path graph is connected, the solution converges to

$$
(6,6,6)^T.
$$

</details>


In [ ]:
u0_new = np.array([0, 6, 12], dtype=float)
for t in [0, 1, 5, 20]:
    print(t, expm(-L*t) @ u0_new)


## 7. Forward Euler as matrix iteration

Forward Euler for $\mathbf x'=A\mathbf x$ is

$$
\mathbf x_{k+1}=(I+hA)\mathbf x_k.
$$


In [ ]:
A_euler = np.array([[-10.0]])
for h in [0.05, 0.15, 0.25]:
    multiplier = 1 + h * (-10)
    print("h=", h, "multiplier=", multiplier, "|multiplier|=", abs(multiplier))


### Similar practice 5

For $x'=-8x$, find the range of step sizes $h>0$ for which forward Euler decays.

<details>
<summary>Solution</summary>

Euler gives

$$
x_{k+1}=(1-8h)x_k.
$$

Decay requires

$$
|1-8h|<1.
$$

Thus

$$
0<h<\frac14.
$$

</details>


## 8. Final reflection

Write a few sentences answering:

1. Why are eigenvectors natural coordinates for $\mathbf x'=A\mathbf x$?
2. Why does $\operatorname{Re}(\lambda)<0$ imply decay?
3. Why does graph heat diffusion preserve the average?
4. Why can forward Euler fail when the exact solution decays?
